# Paper Tooling Notebook Export Tutorial

This notebook demonstrates how to use `paper_tooling.notebook` to:
1. Manage experimental runs with versioning
2. Export figures in multiple formats (PDF, PNG)
3. Export tables in multiple formats (LaTeX, CSV, Excel)
4. Organize artifacts by run_id in paper/ directory structure

## Setup: Import modules

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Import paper_tooling notebook helpers
from paper_tooling.notebook import (
    export_figure,
    export_table,
    start_run,
    get_current_run,
    set_current_run,
    find_project_root,
)

print(f"Paper tooling imported successfully!")

## 1. Find and verify project root

In [ ]:
# Automatically detect the project root
project_root = find_project_root()
print(f"Project root: {project_root}")

# Verify structure
paper_dir = project_root / "paper"
print(f"\nProject structure:")
print(f"  - paper/: {paper_dir.exists()}")
print(f"  - paper/figures/: {(paper_dir / 'figures').exists()}")
print(f"  - paper/tables/: {(paper_dir / 'tables').exists()}")
print(f"  - tooling/paper-tooling/: {(project_root / 'tooling' / 'paper-tooling').exists()}")

## 2. Start a new experimental run

In [ ]:
# Start a new run with an optional label
# Run ID format: YYYY-MM-DD_HHMM_<label>_<git-hash>
run_id = start_run(project_root=project_root, label="baseline-experiment", set_current=True)
print(f"Started new run: {run_id}")

# Verify it's set as current
current = get_current_run(project_root=project_root)
print(f"Current run: {current}")
assert current == run_id

## 3. Create and export a figure

In [ ]:
# Create a simple figure
fig, ax = plt.subplots(figsize=(8, 6))

# Plot some data
x = np.linspace(0, 10, 100)
y1 = np.sin(x)
y2 = np.cos(x)

ax.plot(x, y1, label='sin(x)', linewidth=2)
ax.plot(x, y2, label='cos(x)', linewidth=2)
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('y', fontsize=12)
ax.set_title('Trigonometric Functions', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Export the figure
result = export_figure(
    name="trig_functions",
    fig=fig,
    project_root=project_root,
    formats=["pdf", "png"],
    dpi=300
)

print(f"\nFigure exported successfully!")
print(f"  Run ID: {result['run_id']}")
print(f"  Name: {result['name']}")
print(f"  Paths:")
for fmt, path in result['paths'].items():
    print(f"    {fmt}: {path}")

## 4. Create and export a table

In [ ]:
# Create a sample dataframe with experimental results
data = {
    'Method': ['Baseline', 'Method A', 'Method B', 'Method C'],
    'Accuracy': [0.850, 0.892, 0.905, 0.898],
    'F1-Score': [0.840, 0.888, 0.902, 0.895],
    'Precision': [0.860, 0.900, 0.910, 0.905],
    'Recall': [0.825, 0.880, 0.895, 0.890]
}

df = pd.DataFrame(data)
print("Sample results table:")
print(df.to_string(index=False))

# Export the table
result = export_table(
    name="results_summary",
    df=df,
    project_root=project_root,
    formats=["tex", "csv"],
    index=False
)

print(f"\nTable exported successfully!")
print(f"  Run ID: {result['run_id']}")
print(f"  Name: {result['name']}")
print(f"  Paths:")
for fmt, path in result['paths'].items():
    print(f"    {fmt}: {path}")

## 5. Export multiple figures and tables

In [ ]:
# Export multiple figures from the same run
fig2, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left subplot
axes[0].hist(np.random.randn(1000), bins=30, alpha=0.7, color='blue', edgecolor='black')
axes[0].set_title('Distribution Plot', fontweight='bold')
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Frequency')

# Right subplot
categories = ['A', 'B', 'C', 'D']
values = [23, 45, 56, 78]
axes[1].bar(categories, values, color=['red', 'green', 'blue', 'orange'])
axes[1].set_title('Bar Chart', fontweight='bold')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

# Export this figure
result2 = export_figure(
    name="analysis_plots",
    fig=fig2,
    project_root=project_root,
    formats=["pdf", "png"]
)
print(f"Additional figure exported: {result2['name']}")

# Export another table
df2 = pd.DataFrame({
    'Metric': ['Loss', 'Accuracy', 'Time'],
    'Train': [0.125, 0.920, '2.5s'],
    'Val': [0.150, 0.910, '0.3s']
})

result3 = export_table(
    name="training_stats",
    df=df2,
    project_root=project_root,
    formats=["tex", "csv"]
)
print(f"Additional table exported: {result3['name']}")

## 6. Verify exported artifacts

In [ ]:
from paper_tooling.notebook import get_run_dir

# Get the run directories
fig_run_dir = get_run_dir(project_root=project_root, run_id=run_id, artifact_type="figures")
tbl_run_dir = get_run_dir(project_root=project_root, run_id=run_id, artifact_type="tables")

print(f"Figures directory: {fig_run_dir}")
print(f"Files:")
for file in sorted(fig_run_dir.glob("*")):
    size_kb = file.stat().st_size / 1024
    print(f"  - {file.name} ({size_kb:.1f} KB)")

print(f"\nTables directory: {tbl_run_dir}")
print(f"Files:")
for file in sorted(tbl_run_dir.glob("*")):
    size_kb = file.stat().st_size / 1024
    print(f"  - {file.name} ({size_kb:.1f} KB)")

## 7. Check manifest files

In [ ]:
import json

# Read the figure manifest
manifest_path = fig_run_dir / "manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

print("Figure manifest:")
print(f"  Run ID: {manifest['run_id']}")
print(f"  Created: {manifest['created']}")
print(f"  Exports: {len(manifest['exports'])}")
for i, export in enumerate(manifest['exports'], 1):
    print(f"    {i}. {export['name']} ({export['type']})")
    print(f"       Formats: {', '.join(export['formats'])}")
    print(f"       Timestamp: {export['timestamp']}")

## 8. Starting a second run

In [ ]:
# You can start multiple runs for different experiments
run_id_2 = start_run(project_root=project_root, label="variation-2", set_current=True)
print(f"Started second run: {run_id_2}")

# Current run changes
current = get_current_run(project_root=project_root)
print(f"Current run is now: {current}")

# Each run has its own isolated directory
fig_run_dir_2 = get_run_dir(project_root=project_root, run_id=run_id_2, artifact_type="figures")
print(f"\nSecond run figure directory: {fig_run_dir_2}")
print(f"Same as first run? {fig_run_dir_2 == fig_run_dir}")

## Summary

The `paper_tooling.notebook` module provides:

### Key Functions:
- **`find_project_root()`**: Auto-detect paper repository root
- **`start_run()`**: Create a new versioned run with unique ID
- **`export_figure()`**: Export matplotlib figures (PDF, PNG, etc.)
- **`export_table()`**: Export pandas DataFrames (LaTeX, CSV, Excel, etc.)
- **`get_current_run()`**: Get the active run ID
- **`set_current_run()`**: Switch to a different run
- **`get_run_dir()`**: Get the path for a specific run's artifacts

### Features:
✅ Automatic project root detection  
✅ Run-based organization with versioning (includes git hash)  
✅ Multi-format export (figures and tables)  
✅ Manifest tracking (json) for audit trail  
✅ Lazy imports (only load matplotlib/pandas when needed)  
✅ Nested in paper/figures/runs/ and paper/tables/runs/  

### Directory Structure:
```
project_root/
├── paper/
│   ├── figures/
│   │   ├── CURRENT_RUN.txt
│   │   └── runs/
│   │       └── <run_id>/
│   │           ├── manifest.json
│   │           ├── figure1.pdf
│   │           ├── figure1.png
│   │           └── figure2.pdf
│   └── tables/
│       ├── CURRENT_RUN.txt
│       └── runs/
│           └── <run_id>/
│               ├── manifest.json
│               ├── table1.tex
│               └── table1.csv
└── tooling/
    └── paper-tooling/
```